# 16. Peak mCG motif

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
cd /large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype/
allcools generate-dataset --allc_table /large_storage/zhoulab/zhoujt/project/ENTEx/merged_allc/L1/CGN/allclist_majortype.tsv \
--output_path peak_majortype_CGN.mcds \
--chrom_size_path /large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes \
--obs_dim majortype \
--cpu 32 \
--chunk_size 1 \
--regions peak concat_peak.bed \
--quantifiers peak count CGN


In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from scipy.stats import zscore
from concurrent.futures import ProcessPoolExecutor, as_completed

import anndata
from ALLCools.mcds import MCDS, RegionDS
from ALLCools.clustering import *
from ALLCools.plot import *

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'
peakdir = f'{indir}scATAC/peak/majortype/'
outdir = f'{indir}analysis/mCG_peak_motif/'


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
# L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
L1_meta = L1_meta.drop(['c7'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
import cooler
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
global_mc = pd.read_csv(f'{indir}merged_allc/subtype_globalCG.csv.gz', index_col=0, header=0)
global_mc['L1'] = global_mc.index.str.split('-').str[0]
global_mc.loc[global_mc['L1']=='c7', 'L1'] = 'c35'
global_mc.loc['c7-b1', 'L1'] = 'c36'
global_mc = global_mc.groupby('L1').sum()
global_mc['ratio'] = global_mc['mc'] / global_mc['cov']


In [ ]:
mcds = MCDS.open(f'{peakdir}peak_majortype_CGN.mcds', obs_dim='majortype', var_dim='peak')
mcds


In [ ]:
mcds.add_mc_frac(normalize_per_cell=False)


In [ ]:
data_all = mcds['peak_da_frac'].to_pandas().drop(['c7'], axis=0)
data_all


In [ ]:
peak = mcds[['peak_chrom', 'peak_start', 'peak_end']].to_pandas().drop(['mc_type'], axis=1)
peak.columns = ['chrom', 'start', 'end']
peak

In [ ]:
hypo = (data_all<(global_mc.loc[data_all.index, ['ratio']].values-0.3))
hyper = (data_all>(global_mc.loc[data_all.index, ['ratio']].values+0.1))
print((hypo.sum(axis=0)>0).sum(), (hyper.sum(axis=0)>0).sum())


In [ ]:
print(1)
fig, axes = plt.subplots(2, 1, figsize=(5,4), dpi=300, sharex='all')
ax = axes[0]
sns.histplot(hypo.sum(axis=0), bins=37, binrange=(0,37), ax=ax)
# ax.set_yscale('log')
ax.set_title('# Hypo Methylated Subtypes')
ax = axes[1]
sns.histplot(hyper.sum(axis=0), bins=37, binrange=(0,37), ax=ax)
ax.set_xlim([-1, 38])
ax.set_title('# Hyper Methylated Subtypes')
fig.tight_layout()
# fig.savefig('DMR/DMR_hypohypercount_subtype.pdf', transparent=True)


In [ ]:
peak = peak.reset_index()
peak.index = peak['chrom'].astype(str) + ':' + peak['start'].astype(str) + '-' + peak['end'].astype(str)


In [ ]:
peak_all = []
for ct in L1_meta.index:
    if not os.path.isfile(f'{peakdir}{ct}.bed'):
        continue
    peak_tmp = pd.read_csv(f'{peakdir}{ct}.bed', sep='\t', header=None, index_col=None)
    peak_tmp.columns = ['chrom', 'start', 'end']
    peak_tmp.index = peak_tmp['chrom'].astype(str) + ':' + peak_tmp['start'].astype(str) + '-' + peak_tmp['end'].astype(str)
    peak_tmp['mCG'] = data_all.loc[ct, peak.loc[peak_tmp.index, 'peak'].values].values
    peak_tmp['majortype'] = ct
    peak_all.append(peak_tmp)
    print(ct)
    

In [ ]:
peak_all = pd.concat(peak_all, axis=0)
peak_mid = peak_all.groupby('majortype')['mCG'].median()
peak_ave = peak_all.groupby('majortype')['mCG'].mean()


In [ ]:
# leg_order = peak_ave.sort_values().index
leg_order = global_mc['ratio'].sort_values().index
leg_order = leg_order[leg_order.isin(peak_all['majortype'])]


In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5), dpi=300)
sns.violinplot(peak_all, x='majortype', y='mCG', palette=L1_color, 
               order=leg_order, ax=ax, inner=None)
ax.set_xticks(np.arange(len(leg_order)))
ax.set_xticklabels(leg_order.map(L1_annot), rotation=90)
fig.tight_layout()
fig.savefig(f'{outdir}peak_mCG_violin.pdf', transparent=True)


In [ ]:
mid = (peak_all['start'] + peak_all['end']) // 2
peak_all['start'] = mid - 500
peak_all['end'] = mid + 500
peak_all = peak_all.loc[(peak_all['start']>0) & (peak_all['end']<peak_all['chrom'].map(chrom_sizes))]


In [ ]:
for ct,peak_tmp in peak_all.groupby('majortype'):
    os.makedirs(f'{outdir}{ct}/bed/hypo/', exist_ok=True)
    os.makedirs(f'{outdir}{ct}/bed/hyper/', exist_ok=True)
    peak_hypo = peak_tmp.loc[peak_tmp['mCG']<0.1]
    peak_hyper = peak_tmp.loc[peak_tmp['mCG']>(global_mc.loc[ct, 'ratio']+0.1)]
    peak_hypo[['chrom', 'start', 'end']].to_csv(f'{outdir}{ct}/bed/hypo/{ct}.bed', sep='\t', header=False, index=False)
    peak_hyper[['chrom', 'start', 'end']].to_csv(f'{outdir}{ct}/bed/hyper/{ct}.bed', sep='\t', header=False, index=False)
    print(ct, peak_hypo.shape[0], peak_hyper.shape[0])
    

In [ ]:
## summit/loop ATAC peak enriched motif

for i in `seq 1 35`; do (
ct=c${i}
dbdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif
outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/mCG_peak_motif
if test -e "${outdir}/${ct}/bed/hypo/${ct}.bed"; then 
scenicplus grn_inference motif_enrichment_cistarget \
    --region_set_folder ${outdir}/${ct}/bed \
    --cistarget_db_fname ${dbdir}/${ct}/merge_peak.regions_vs_motifs.rankings.feather \
    --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
    --temp_dir ${outdir}/${ct}/tmp \
    --species "homo_sapiens" \
    --fr_overlap_w_ctx_db 0.9 \
    --auc_threshold 0.005 \
    --nes_threshold 3.0 \
    --rank_threshold 0.05 \
    --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
    --annotation_version "v10nr_clust" \
    --motif_similarity_fdr 0.001 \
    --orthologous_identity_threshold 0.0 \
    --annotations_to_use "Direct_annot Orthology_annot" \
    --write_html \
    --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
    --n_cpu 32
fi
)
done


In [ ]:
motif = pd.read_csv(f'{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl', sep='\t', header=0, index_col=0)
motif

In [ ]:
/large_storage/zhoulab/ref/motif_databases/JASPAR/JASPAR2024_CORE_non-redundant_pfms.meme


In [ ]:
motif = motif.loc[motif['description'].str.split(' ').str[0]=='gene']
motif.shape

In [ ]:
motif_list = pd.read_csv(f'{REF_ROOT}/aertslab_motif_colleciton/motifs.txt', index_col=0, header=None).index
motif_list = motif_list.str.replace('.cb','')


In [ ]:
motif_list.isin(motif.index).sum()

In [ ]:
motif = motif.loc[motif.index.isin(motif_list)]
print(motif.shape)

In [ ]:
motif.loc[(motif['gene_name']=='CTCF') & (motif['description']=='gene is directly annotated')].index.unique()

In [ ]:
#tf2call['Call'].unique()
callmap = {'MethylPlus':'MethylPlus', 
           'MethylMinus':'MethylMinus', 
           'Little effect and MethylMinus':'MethylMinus',
           'MethylMinus and MethylPlus':'no info', 
           'Little effect and MethylPlus':'MethylPlus',
           'MethylPlus and Little effect':'MethylPlus', 
           'MethylMinus and Little effect':'MethylMinus',
           'MethylPlus and MethylMinus':'no info', 
           'Little effect':'Little effect', 
           'inconclusive':'no info'
          }

In [ ]:
motif_mc_annot = pd.read_csv(f'{REF_ROOT}/yin_2017_motif_type.csv', header=0, index_col=0)
motif_mc_annot = motif_mc_annot.loc[~motif_mc_annot.index.isna()]
motif_mc_annot['Call'] = motif_mc_annot['Call'].map(callmap)
motif_mc_annot = motif_mc_annot.loc[motif_mc_annot.index.isin(motif['gene_name'])]
motif_mc_annot

In [ ]:
motif_mc_dict = pd.Series('no info', index=motif['gene_name'].unique())
motif_mc_dict.loc[motif_mc_annot.index] = motif_mc_annot['Call'].copy()
motif_mc_dict.value_counts()

In [ ]:
col = ['Unnamed: 0', 'Region_set', 'NES', 'AUC', 'Rank_at_max',
       'Motif_hits']
count = {}
for ct in L1_meta.index:
    file_path = f'{outdir}{ct}/ctx_results.html'
    if not os.path.isfile(f'{peakdir}{ct}.bed'):
        continue
    tmp = pd.read_html(file_path)[0][col].set_index('Unnamed: 0')
    tmp.index.name = '#motif_id'
    tmp['region'] = tmp['Region_set'].str.split('_').str[0]
    motif_gene = motif.loc[motif.index.isin(tmp.index[tmp['region']=='hyper']), 'gene_name'].unique()
    count[ct] = motif_mc_dict.loc[motif_gene].value_counts()

count = pd.DataFrame(count).fillna(0)


In [ ]:
count

In [ ]:
motif['gene_name'].unique().shape

In [ ]:
motif_mc_annot['Call'].value_counts()

In [ ]:
motif_mc_dict.value_counts()

In [ ]:
count = count.loc[['MethylMinus', 'MethylPlus']]
count = count.loc[:, count.sum(axis=0)>0]
motif_mc_dict = motif_mc_dict.loc[motif_mc_dict.isin(['MethylMinus', 'MethylPlus'])]
o2e = (count / count.sum(axis=0)).T / motif_mc_dict.value_counts() * motif_mc_dict.shape[0]
sns.clustermap(np.log2(o2e+0.01), vmin=-1, vmax=1)


## AME HOCOMOCO

In [ ]:
for ct in $(ls -d c*); do (
bedtools getfasta -fi /large_storage/zhoulab/ref/hg38/fasta/hg38.fa -fo ${ct}/bed/hypo/${ct}.fa -bed ${ct}/bed/hypo/${ct}.bed
bedtools getfasta -fi /large_storage/zhoulab/ref/hg38/fasta/hg38.fa -fo ${ct}/bed/hyper/${ct}.fa -bed ${ct}/bed/hyper/${ct}.bed
ame --control ${ct}/bed/hyper/${ct}.fa --o ${ct}/out_hypo ${ct}/bed/hypo/${ct}.fa /large_storage/zhoulab/ref/motif_databases/HUMAN/HOCOMOCOv11_core_HUMAN_mono_meme_format.meme
ame --control ${ct}/bed/hypo/${ct}.fa --o ${ct}/out_hyper ${ct}/bed/hyper/${ct}.fa /large_storage/zhoulab/ref/motif_databases/HUMAN/HOCOMOCOv11_core_HUMAN_mono_meme_format.meme
) & 
done


In [ ]:
indir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis
for ct in $(ls -d c*); do (
bedtools getfasta -fi /large_storage/zhoulab/ref/hg38/fasta/hg38.fa -fo ${ct}/bed/${ct}.fa -bed ${indir}/loop_peak_motif/${ct}/merge_peak.slop1kb.bed
ame --control ${ct}/bed/${ct}.fa --o ${ct}/out_hyper_all ${ct}/bed/hyper/${ct}.fa /large_storage/zhoulab/ref/motif_databases/HUMAN/HOCOMOCOv11_core_HUMAN_mono_meme_format.meme
ame --control ${ct}/bed/${ct}.fa --o ${ct}/out_hypo_all ${ct}/bed/hypo/${ct}.fa /large_storage/zhoulab/ref/motif_databases/HUMAN/HOCOMOCOv11_core_HUMAN_mono_meme_format.meme
) & 
done


In [ ]:
motif_list = pd.read_csv(f'{outdir}HOCOMOCOv11_core_motif_list.txt', index_col=0, header=None).index
motif_list = motif_list.str.split(' ').str[1].str.split('_').str[0]
motif_list

In [ ]:
motif_mc_annot = pd.read_csv(f'{REF_ROOT}/yin_2017_motif_type.csv', header=0, index_col=0)
motif_mc_annot = motif_mc_annot.loc[~motif_mc_annot.index.isna()]
motif_mc_annot['Call'] = motif_mc_annot['Call'].map(callmap)
motif_mc_annot = motif_mc_annot.loc[motif_mc_annot.index.isin(motif_list)]
motif_mc_dict = pd.Series('no info', index=motif_list.unique())
motif_mc_dict.loc[motif_mc_annot.index] = motif_mc_annot['Call'].copy()


In [ ]:
count = {}
pv = {}
for ct in L1_meta.index:
    file_path = f'{outdir}{ct}/out_hypo/ame.tsv'
    if not os.path.isfile(file_path):
        continue
    try:
        tmp = pd.read_csv(file_path, sep='\t', comment='#')
    except pd.errors.EmptyDataError:
        print(f'{ct} empty')
        continue
    tmp['gene_name'] = tmp['motif_ID'].str.split('_').str[0]
    tmp = tmp.loc[tmp['gene_name'].isin(motif_mc_dict.index) & (tmp['adj_p-value']<1e-2)]
    count[ct] = tmp['gene_name'].map(motif_mc_dict).value_counts()
    pv[ct] = tmp.set_index('motif_ID')['adj_p-value']

count = pd.DataFrame(count).fillna(0)
pv = pd.DataFrame(pv).fillna(1)


In [ ]:
sns.clustermap(np.clip(-np.log10(pv), a_min=0, a_max=100), xticklabels=pv.columns.map(L1_annot))

In [ ]:
sns.clustermap(np.clip(-np.log10(pv), a_min=0, a_max=100), xticklabels=pv.columns.map(L1_annot))

In [ ]:
motif_mc_dict.value_counts()

In [ ]:
count

In [ ]:
count = count.loc[['MethylMinus', 'MethylPlus']]
motif_mc_dict = motif_mc_dict.loc[motif_mc_dict.isin(['MethylMinus', 'MethylPlus'])]
o2e = (count / count.sum(axis=0)).T / motif_mc_dict.value_counts() * motif_mc_dict.shape[0]
# sns.clustermap(np.log2(o2e+0.01), vmin=-1, vmax=1, cmap='vlag')
fig, ax = plt.subplots(figsize=(2,5), dpi=300)
ax.imshow(np.log2(o2e+0.01), vmin=-1, vmax=1, cmap='vlag', aspect='auto')
ax.set_yticks(np.arange(o2e.shape[0]))
ax.set_yticklabels(o2e.index.map(L1_annot))
ax.set_xticks([0,1])
ax.set_xticklabels(o2e.columns, rotation=90)
ax.set_title('Hypo-mCG Peak')
fig.tight_layout()
fig.savefig(f'{outdir}hypo_motif.pdf', dpi=300)


In [ ]:
count = count.loc[['MethylMinus', 'MethylPlus']]
motif_mc_dict = motif_mc_dict.loc[motif_mc_dict.isin(['MethylMinus', 'MethylPlus'])]
o2e = (count / count.sum(axis=0)).T / motif_mc_dict.value_counts() * motif_mc_dict.shape[0]
sns.clustermap(np.log2(o2e+0.01), vmin=-1, vmax=1, cmap='vlag')


In [ ]:
count = count.loc[['MethylMinus', 'MethylPlus']]
motif_mc_dict = motif_mc_dict.loc[motif_mc_dict.isin(['MethylMinus', 'MethylPlus'])]
o2e = (count / count.sum(axis=0)).T / motif_mc_dict.value_counts() * motif_mc_dict.shape[0]
# sns.clustermap(np.log2(o2e+0.01), vmin=-1, vmax=1, cmap='vlag')
fig, ax = plt.subplots(figsize=(2,5), dpi=300)
ax.imshow(np.log2(o2e+0.01), vmin=-1, vmax=1, cmap='vlag', aspect='auto')
ax.set_yticks(np.arange(o2e.shape[0]))
ax.set_yticklabels(o2e.index.map(L1_annot))
ax.set_xticks([0,1])
ax.set_xticklabels(o2e.columns, rotation=90)
ax.set_title('Hyper-mCG Peak')
fig.tight_layout()
fig.savefig(f'{outdir}hyper_motif.pdf', dpi=300)


In [ ]:
count = count.loc[['MethylMinus', 'MethylPlus']]
motif_mc_dict = motif_mc_dict.loc[motif_mc_dict.isin(['MethylMinus', 'MethylPlus'])]
o2e = (count / count.sum(axis=0)).T / motif_mc_dict.value_counts() * motif_mc_dict.shape[0]
sns.clustermap(np.log2(o2e+0.01), vmin=-1, vmax=1, cmap='vlag')
